# 🌬️ Ćwiczenie 12: Structured Streaming — System monitoringu jakości powietrza

**Kurs:** Python w Zastosowaniach Big Data (Studia stacjonarne)
**Zajęcia nr:** 12 (12.01.2027)
**Szacowany czas:** ⏱️ ~90 minut
**Poziom trudności:** ⭐⭐⭐⭐ (Zaawansowany)

## 📚 Cele nauki

Pracujesz jako inżynier danych w miejskim systemie **EcoMonitor** 🌬️ — monitoringu jakości powietrza w czasie rzeczywistym. Miasto rozmieściło ~30 czujników mierzących PM2.5, PM10, temperaturę i wilgotność. Po ukończeniu tego notebooka będziesz potrafić:

1. Symulować strumień danych IoT generatorem plików (przybliżenie Kafka producer w Colab)
2. Budować strumienie z `readStream` / `writeStream` w Spark Structured Streaming
3. Stosować **windowed aggregations** z watermarkingiem dla spóźnionych danych
4. Porównywać output modes: `append` vs `complete`
5. Bezpiecznie zatrzymywać i wznawiać streaming query
6. Rozumieć, jak wyglądałby prawdziwy producer/consumer Apache Kafka

## 📋 Wymagania wstępne
- Znajomość podstaw PySpark DataFrame API (`select`, `filter`, `groupBy`)
- Podstawowa wiedza o oknach czasowych (`window()`)

---
> 💡 **Tip:** Uruchamiaj komórki po kolei (`Shift+Enter`). Streaming query trzeba czasem chwilę poczekać, aż przetworzy pierwszy mikro-batch!

In [ ]:
# @title 🔧 Setup — Run this cell first!

!pip install -q pyspark

import os
import json
import random
import shutil
import tempfile
import time
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg, max as _max
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, TimestampType
)

spark = (SparkSession.builder
    .appName("EcoMonitorStreaming")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

random.seed(42)  # Reprodukowalność 🎲

print("✅ Setup complete! Spark w wersji:", spark.version)

## 📖 Quick Refresher

**Structured Streaming** traktuje strumień danych jako **unbounded table** (nieskończoną tabelę) — nowe dane to po prostu kolejne wiersze. Piszesz zwykły kod DataFrame/SQL, a Spark sam dba o przyrostowe przetwarzanie.

W prawdziwym środowisku produkcyjnym dane wpływałyby przez **Apache Kafka** (topic → partitions → consumer). W Google Colab nie mamy dostępu do klastra Kafki, więc symulujemy strumień **generatorem plików JSON** zapisywanych do folderu — Spark odczytuje je dokładnie tak samo przez `readStream`!

### Kluczowe pojęcia

| Pojęcie | Opis | Przykład |
|---------|------|----------|
| `readStream` | Odczyt danych jako strumienia (unbounded table) | `spark.readStream.schema(s).json(path)` |
| `writeStream` | Zapis wyniku strumienia | `df.writeStream.outputMode("complete")` |
| Window | Agregacja w oknie czasowym | `window("timestamp", "5 minutes")` |
| Watermark | Próg tolerancji na spóźnione dane | `withWatermark("timestamp", "10 minutes")` |
| Output mode | Sposób zapisu wyniku agregacji | `append` / `update` / `complete` |

In [ ]:
# @title 📡 Generator strumienia — symulacja ~30 czujników jakości powietrza

STREAM_DIR = tempfile.mkdtemp(prefix="ecomonitor_stream_")
CHECKPOINT_DIR = tempfile.mkdtemp(prefix="ecomonitor_checkpoint_")
print(f"📁 Folder strumienia: {STREAM_DIR}")

SENSOR_IDS = [f"sensor_{i:02d}" for i in range(1, 31)]  # 30 czujników w mieście


def generuj_mikro_batch(batch_id: int, base_time: datetime, liczba_odczytow: int = 30) -> str:
    """Symuluje jeden mikro-batch odczytów z czujników (jak Kafka producer)."""
    odczyty = []
    for sensor_id in SENSOR_IDS[:liczba_odczytow]:
        # Bazowy poziom PM2.5 z losowym szumem + okazjonalny "skok" smogu
        pm25 = max(0.0, random.gauss(35, 15))
        if random.random() < 0.05:  # 5% szans na skok zanieczyszczenia
            pm25 += random.uniform(60, 120)
        odczyty.append({
            "sensor_id": sensor_id,
            "pm25": round(pm25, 1),
            "pm10": round(pm25 * random.uniform(1.1, 1.6), 1),
            "temperatura": round(random.gauss(5, 4), 1),
            "wilgotnosc": round(random.uniform(40, 90), 1),
            "timestamp": base_time.strftime("%Y-%m-%d %H:%M:%S"),
        })

    plik = os.path.join(STREAM_DIR, f"batch_{batch_id:04d}.json")
    with open(plik, "w", encoding="utf-8") as f:
        for odczyt in odczyty:
            f.write(json.dumps(odczyt) + "\n")
    return plik


# Wygeneruj pierwsze mikro-batche "historyczne" — symulacja kilkunastu minut danych
start_time = datetime(2027, 1, 12, 10, 0, 0)
for i in range(6):
    generuj_mikro_batch(i, start_time + timedelta(minutes=i))

print(f"✅ Wygenerowano 6 mikro-batchy w {STREAM_DIR}")

## 🎓 Guided Example — Windowed average PM2.5 per sensor

**Zadanie:** Czytamy strumień plików JSON, liczymy średnią PM2.5 per czujnik w oknach 5-minutowych, zapisujemy wynik do tabeli `memory` i odpytujemy ją przez SQL.

In [ ]:
# @title 🎓 Guided Example — Run and study this

schemat_czujnikow = StructType([
    StructField("sensor_id", StringType()),
    StructField("pm25", DoubleType()),
    StructField("pm10", DoubleType()),
    StructField("temperatura", DoubleType()),
    StructField("wilgotnosc", DoubleType()),
    StructField("timestamp", TimestampType()),
])

# 1️⃣ Czytamy strumień plików JSON (docelowo byłby to topic Kafka) 📥
strumien = (spark.readStream
    .schema(schemat_czujnikow)
    .option("maxFilesPerTrigger", 1)
    .json(STREAM_DIR))

# 2️⃣ Windowed aggregation — średnia PM2.5 per czujnik, okno 5 minut
srednie_pm25 = (strumien
    .withWatermark("timestamp", "10 minutes")
    .groupBy(
        window(col("timestamp"), "5 minutes"),
        col("sensor_id"),
    )
    .agg(avg("pm25").alias("srednia_pm25")))

# 3️⃣ Zapis do tabeli w pamięci — memory sink, do odpytywania SQL 🧠
query = (srednie_pm25.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("srednie_pm25_tabela")
    .start())

time.sleep(8)  # Dajemy Sparkowi chwilę na przetworzenie mikro-batchy

wynik = spark.sql("SELECT * FROM srednie_pm25_tabela ORDER BY window, sensor_id")
print(f"📊 Liczba wierszy w tabeli wyników: {wynik.count()}")
wynik.show(10, truncate=False)

---
## ✏️ Ćwiczenie 1: Wykrywanie przekroczeń normy PM2.5 (alerty)

**Trudność:** ⭐

Norma dla PM2.5 wynosi 100 µg/m³ (poziom alarmowy). Uzupełnij kod, aby przefiltrować `wynik` (tabelę z guided example) i znaleźć wiersze, w których `srednia_pm25` przekracza próg alertu.

**Oczekiwany wynik:** DataFrame `alerty_df` zawierający tylko wiersze przekraczające próg.

> 💡 **Hint:** Użyj `.filter(col("srednia_pm25") > PROG_ALERTU)` na DataFrame `wynik`.

In [ ]:
# ✏️ Ćwiczenie 1: Wykrywanie przekroczeń normy PM2.5

PROG_ALERTU = 100.0

# TODO: przefiltruj `wynik`, zostawiając tylko wiersze z srednia_pm25 > PROG_ALERTU
alerty_df = ...  # Replace ... with your code

print(f"🚨 Liczba alertów: {alerty_df.count()}")
alerty_df.show(truncate=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 1 (Run to check your answer)

try:
    assert alerty_df is not ..., "❌ Nie uzupełniłeś/aś alerty_df"
    zebrane = alerty_df.select("srednia_pm25").collect()
    assert all(row["srednia_pm25"] > PROG_ALERTU for row in zebrane), (
        "❌ Znaleziono wiersze poniżej progu alertu"
    )
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 2: Rolling average — sliding window

**Trudność:** ⭐⭐

Zamiast tumbling window (okna nienakładające się), zbuduj **sliding window**: okno 5-minutowe, przesuwane co 1 minutę. Dzięki temu średnia PM2.5 aktualizuje się częściej.

**Oczekiwany wynik:** nowy streaming DataFrame `srednia_sliding` zagregowany przez `window("timestamp", "5 minutes", "1 minute")`.

> 💡 **Hint:** `window()` przyjmuje trzeci argument opcjonalny — długość przesunięcia (slide duration).

In [ ]:
# ✏️ Ćwiczenie 2: Sliding window rolling average

# TODO: zbuduj agregację ze sliding window (5 minut, slide 1 minuta)
srednia_sliding = ...  # Replace ... with your code, np.:
# srednia_sliding = (strumien
#     .withWatermark("timestamp", "10 minutes")
#     .groupBy(window(col("timestamp"), "5 minutes", "1 minute"), col("sensor_id"))
#     .agg(avg("pm25").alias("srednia_pm25")))

print("Typ obiektu:", type(srednia_sliding))

In [ ]:
# @title ✅ Test — Ćwiczenie 2 (Run to check your answer)

try:
    assert srednia_sliding is not ..., "❌ Nie uzupełniłeś/aś srednia_sliding"
    assert srednia_sliding.isStreaming, "❌ srednia_sliding powinien być streaming DataFrame"
    kolumny = srednia_sliding.columns
    assert "window" in kolumny and "srednia_pm25" in kolumny, (
        "❌ Brakuje kolumn 'window' lub 'srednia_pm25'"
    )
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 3: Watermarking dla spóźnionych odczytów

**Trudność:** ⭐⭐

Wygeneruj dodatkowy mikro-batch odczytów ze **spóźnionym timestampem** (np. 20 minut wstecz od ostatniego wygenerowanego batcha) i sprawdź, czy watermark (10 minut) go odrzuci.

**Oczekiwany wynik:** funkcja `generuj_spozniony_batch()` zwracająca ścieżkę do wygenerowanego pliku ze spóźnionymi danymi.

> 💡 **Hint:** Skopiuj logikę z `generuj_mikro_batch`, ale ustaw `base_time` na dużo wcześniejszą godzinę.

In [ ]:
# ✏️ Ćwiczenie 3: Watermarking dla spóźnionych odczytów

def generuj_spozniony_batch(batch_id: int) -> str:
    """Generuje mikro-batch ze spóźnionym timestampem (20 minut wstecz)."""
    # TODO: użyj (start_time - timedelta(minutes=20)) jako base_time
    # i wywołaj generuj_mikro_batch z tym czasem
    opozniony_czas = ...  # Replace ... with your code
    return generuj_mikro_batch(batch_id, opozniony_czas)


sciezka_spozniona = generuj_spozniony_batch(100)
print(f"📁 Wygenerowano spóźniony batch: {sciezka_spozniona}")

In [ ]:
# @title ✅ Test — Ćwiczenie 3 (Run to check your answer)

try:
    assert opozniony_czas is not ..., "❌ Nie uzupełniłeś/aś opozniony_czas"
    roznica = (start_time - opozniony_czas).total_seconds() / 60
    assert abs(roznica - 20) < 0.01, f"❌ Oczekiwano 20 minut różnicy, otrzymano {roznica}"
    assert os.path.exists(sciezka_spozniona), "❌ Plik nie został utworzony"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 4: Output modes — append vs complete

**Trudność:** ⭐⭐⭐

Uruchom nowe streaming query z `outputMode("append")` na strumieniu **bez agregacji** (surowe odczyty przekraczające próg PM2.5), a następnie porównaj je z podejściem `"complete"` z guided example.

**Oczekiwany wynik:** zmienna `query_append` — uruchomione query zapisujące surowe alerty do tabeli `memory` w trybie `append`.

> 💡 **Hint:** `outputMode("append")` działa bez watermarku tylko dla operacji bez stanu (np. `filter`, bez `groupBy`).

In [ ]:
# ✏️ Ćwiczenie 4: Output modes — append vs complete

surowe_alerty = strumien.filter(col("pm25") > PROG_ALERTU)

# TODO: uruchom writeStream w outputMode "append", format "memory", queryName "surowe_alerty_tabela"
query_append = ...  # Replace ... with your code

time.sleep(8)
print(spark.sql("SELECT COUNT(*) AS liczba_alertow FROM surowe_alerty_tabela").collect())

In [ ]:
# @title ✅ Test — Ćwiczenie 4 (Run to check your answer)

try:
    assert query_append is not ..., "❌ Nie uzupełniłeś/aś query_append"
    assert query_append.isActive, "❌ query_append powinno być aktywne"
    assert query_append.name == "surowe_alerty_tabela", "❌ Niepoprawna queryName"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 5: Agregacja per para PM2.5/PM10

**Trudność:** ⭐⭐⭐

Rozszerz agregację z guided example, aby liczyła również **maksimum PM10** per czujnik w tym samym oknie 5-minutowym (obok średniej PM2.5).

**Oczekiwany wynik:** streaming DataFrame `srednie_i_max` zawierający kolumny `srednia_pm25` oraz `max_pm10`.

> 💡 **Hint:** `.agg()` przyjmuje wiele kolumn agregujących naraz, np. `avg(...).alias(...), max(...).alias(...)`.

In [ ]:
# ✏️ Ćwiczenie 5: Agregacja PM2.5 (avg) i PM10 (max) razem

# TODO: zbuduj agregację z avg(pm25) i max(pm10) w tym samym oknie 5-minutowym
srednie_i_max = ...  # Replace ... with your code

print("Kolumny:", srednie_i_max.columns if srednie_i_max is not ... else "brak")

In [ ]:
# @title ✅ Test — Ćwiczenie 5 (Run to check your answer)

try:
    assert srednie_i_max is not ..., "❌ Nie uzupełniłeś/aś srednie_i_max"
    kolumny = srednie_i_max.columns
    assert "srednia_pm25" in kolumny, "❌ Brakuje kolumny 'srednia_pm25'"
    assert "max_pm10" in kolumny, "❌ Brakuje kolumny 'max_pm10'"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 6: Bezpieczne zatrzymanie i wznowienie query

**Trudność:** ⭐⭐⭐

Zatrzymaj query `query` z guided example (`.stop()`), poczekaj chwilę, a następnie uruchom je ponownie na tym samym strumieniu z tą samą `queryName`. Sprawdź, że po ponownym uruchomieniu query jest aktywne.

**Oczekiwany wynik:** zmienna `query_restarted` — nowe, aktywne streaming query po restarcie.

> 💡 **Hint:** Zawsze najpierw wywołaj `.stop()` na starym query, zanim uruchomisz nowe o tej samej nazwie.

In [ ]:
# ✏️ Ćwiczenie 6: Bezpieczne zatrzymanie i wznowienie query

# TODO: zatrzymaj `query` metodą .stop()
...  # Replace ... with your code

time.sleep(2)

# TODO: uruchom ponownie tę samą agregację (srednie_pm25) writeStream,
# outputMode("complete"), format("memory"), queryName("srednie_pm25_tabela")
query_restarted = ...  # Replace ... with your code

time.sleep(5)
print("Aktywne po restarcie:", query_restarted.isActive if query_restarted is not ... else "brak")

In [ ]:
# @title ✅ Test — Ćwiczenie 6 (Run to check your answer)

try:
    assert query_restarted is not ..., "❌ Nie uzupełniłeś/aś query_restarted"
    assert not query.isActive, "❌ Stare query powinno być zatrzymane"
    assert query_restarted.isActive, "❌ Nowe query powinno być aktywne"
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## ✏️ Ćwiczenie 7: Analiza wyników przez SQL

**Trudność:** ⭐⭐

Napisz zapytanie SQL na tabeli `srednie_pm25_tabela`, które zwraca **czujnik z najwyższą średnią PM2.5** (najbardziej zanieczyszczoną lokalizacją).

**Oczekiwany wynik:** DataFrame `najgorszy_czujnik` z dokładnie jednym wierszem.

> 💡 **Hint:** `ORDER BY srednia_pm25 DESC LIMIT 1`.

In [ ]:
# ✏️ Ćwiczenie 7: Najbardziej zanieczyszczony czujnik

# TODO: napisz zapytanie SQL zwracające czujnik z najwyższą srednia_pm25
najgorszy_czujnik = ...  # Replace ... with your code
# np. spark.sql("SELECT * FROM srednie_pm25_tabela ORDER BY srednia_pm25 DESC LIMIT 1")

if najgorszy_czujnik is not ...:
    najgorszy_czujnik.show(truncate=False)

In [ ]:
# @title ✅ Test — Ćwiczenie 7 (Run to check your answer)

try:
    assert najgorszy_czujnik is not ..., "❌ Nie uzupełniłeś/aś najgorszy_czujnik"
    assert najgorszy_czujnik.count() == 1, (
        f"❌ Oczekiwano 1 wiersza, otrzymano {najgorszy_czujnik.count()}"
    )
    print("✅ Wszystkie testy przeszły! Świetna robota! 🎉")
except AssertionError as e:
    print(str(e))
    print("Spróbuj jeszcze raz! Dasz radę 💪")

---
## 🏆 Bonus Challenge (Optional): Prawdziwa Kafka zamiast symulacji plikowej

W produkcji zamiast generatora plików JSON użylibyśmy **prawdziwego producenta i konsumenta Apache Kafka**. Poniższy kod jest **wyłącznie referencyjny** — wymaga działającego brokera Kafka i biblioteki `kafka-python`, więc **nie uruchamiamy go w tym notebooku**.

```python
# 🔧 Przykładowy Kafka producer (NIE URUCHAMIAĆ bez działającego brokera!)
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)

odczyt = {
    "sensor_id": "sensor_01",
    "pm25": 42.3,
    "pm10": 58.1,
    "temperatura": 4.2,
    "wilgotnosc": 71.5,
    "timestamp": "2027-01-12 10:05:00",
}
producer.send("powietrze-pm25", value=odczyt)
producer.flush()

# 🔧 Odpowiadający Spark readStream z Kafki (produkcyjny)
df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "powietrze-pm25")
    .load())
```

**Pytanie do namysłu:** Jakie zmiany w kodzie musiałbyś/musiałabyś wprowadzić, przechodząc z file source na Kafkę? (Podpowiedź: głównie zmienia się tylko `.format(...)` i opcje źródła — reszta pipeline'u z `window`/`watermark`/`writeStream` zostaje bez zmian!)

In [ ]:
# @title 🧹 Sprzątanie — zatrzymaj wszystkie aktywne query i usuń pliki tymczasowe

for q in spark.streams.active:
    q.stop()

shutil.rmtree(STREAM_DIR, ignore_errors=True)
shutil.rmtree(CHECKPOINT_DIR, ignore_errors=True)

print("✅ Wszystkie streaming queries zatrzymane, pliki tymczasowe usunięte.")

---
## 📝 Summary

Świetna robota! 🎉 To był **ostatni notebook** kursu — a przećwiczyłeś/aś w nim:

- ✅ Symulację strumienia danych IoT jako przybliżenie Kafka producer
- ✅ `readStream` / `writeStream` w Spark Structured Streaming
- ✅ Windowed aggregations (tumbling i sliding windows)
- ✅ Watermarking dla spóźnionych danych
- ✅ Output modes: `append` vs `complete`
- ✅ Bezpieczne zatrzymywanie i wznawianie streaming query
- ✅ Koncepcyjne podstawy prawdziwej integracji z Apache Kafka

### 🎓 Cały kurs w pigułce

Od statystyki opisowej (Wykład 1) przez Hadoop, Spark RDD/SQL, feature engineering, MLlib, Cloud Computing — aż po dzisiejszy streaming. Przeszliście całą drogę od danych statycznych do przetwarzania w czasie rzeczywistym! 🌊

### 📚 Dalsze czytanie
- [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)
- [Spark + Kafka Integration Guide](https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html)

### 🔮 Co dalej?
**Następne zajęcia (19.01.2027) to Egzamin teoretyczny** (70% oceny, Wykłady 1-12). Powodzenia! 💪🎓